# Plotting recommendations for Chem 4410 - Part 2

I have presented some examples of data plots with linear and non-linear regression data analysis in previous classes and will present many more as we proceed. Over the years my methodology has evolved as I learn more about the tools available. You will see many approaches used to reach the exact same conclusions. This document supports a series of *Python* notebooks that present my current methods. Feel free to steal all this work and use it for yourself. All I ask is that you give credit when appropriate.

The code below will load in tools and set some global variables that will be used in all the code blocks below. Much of that code is specific for running this notebook in Colab. We need to install packages that are not included in Colab and load in a data file and a plotting style file to use later.

In [ ]:
# Install packages only if needed
# Coogle Colab does not include every Python package
#   
try:                  # if the package exists import it, otherwise install it
    import lmfit
except ImportError:
    !pip -q install lmfit
    import lmfit

try:
    import uncertainties
except ImportError:
    !pip -q install uncertainties
    import uncertainties

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import uncertainties as un
from uncertainties import unumpy as unp

from matplotlib.ticker import FormatStrFormatter
from scipy.stats import t

from pathlib import Path

Path("plots").mkdir(exist_ok=True)
Path("data").mkdir(exist_ok=True)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path("tufte.mplstyle").exists():
    !wget -q https://raw.githubusercontent.com/blinkletter/Chem4410Webbook/main/book/styles/tufte.mplstyle

plt.rcdefaults()
plt.rcParams["figure.figsize"] = (4, 3.5)    # set some default style paramers for plots

print("Setup complete")


## A Curved Plot

In the [previous notebook](01_SimplePlots.ipynb), we explored a linear fit for a first order reaction. When the concentration became small, the experimental error resulted in negative values, which could not be included in the linearize model. This is one reason why we should consider reducing our use of the straight line. We don't need to stay within the limts of *Excel*; we can use *Python*.

The integrated rate equation for a first order reaction is...

$$A_t - A_\infty = \left(A_0 - A_\infty \right)e^{-kt} $$

...and thats all we need.

We will assume that the concentration of absorbance goes to zero (we are subtracting the background) as time progresses. So the equation is...

$$A_t = A_0 e^{-kt} $$

### Get the Data

We could make random again and I will repeat the code used previously to make fake data for a first order reaction where we are following the dissappearance of the reactant. the code below will make random data and write a datafile.



In [ ]:
# Generate random Data for first-order chemical kinetics

# True reaction parameters
A0 = 0.80       # Initial absorbance at 254nm (M)
k = 0.150       # First-order rate constant (min^-1)
epsilon = 300   # molar absorbtivity of reactant at 254 nm (M^-1)

start = 1       # time of first observation (min)
end = 40        # time of last observation (min)

n = 10         # Number of observations
sigma = 0.03    # Standard deviation of measurement error (Abs)

flag = True    # set flag to True to fix endpoints; set to False to fit all three valiables

# Generate time points
time = np.linspace(start, end, n)  # time points of observations

# Calculate true values
A_true = A0 * np.exp(-k * time) # calc

# Add experimental error
np.random.seed(54321)          # For reproducibility - will always get the same series of "random" from a given seed


A_obs = A_true + np.random.normal(
    loc=0,
    scale=sigma,
    size=len(A_true)
)

# Print data table
if False:                     # set to True to print out data, False to skip
    print(" Time (min)   Abs A observed (254nm)")
    for t, a in zip(time, A_obs):
        print(f"{t:10.2f}   {a:12.4f}")

# Create DataFrame
df = pd.DataFrame({
    "Time_min": time,
    "Abs_obs": A_obs
})

# Write to CSV
filename = "data/first_order_kinetics2.csv"
df.to_csv(filename, 
          index=False,
          float_format="%.3f")

print(f"\n Data written to {filename}")    # 



### Load the Data

The code below will import the text from the csv file and create a new dataframe containing the data. We can start here so we use the same data again and again. We can reuse the code above if you want new data. perhaps you want to change the standard deviation of the random error.

there is no reason to read back in data that we have just created, but I want this notebook to be generally useful. You can also use this code to read in a csv file of your own experimental data. 

In [ ]:
df = pd.read_csv(filename)   # read in the file and convert it to a DataFrame using pandas read_csv tool
print(df.head())             # print just the first few lines

### Calculations

I can use the data directly and convert the units of the result or I can convert the units now and then use the data to produce a result with the desired units. Currently the absorbance of the starting material is being followed. the following code will convert it to concentration and we will have a concentration vs time plot.

We will use a molar absorptivity constant of $300\;M^{-1}$. I will choose to use $mM$ for the concenbtration (less zeros to deal with).

In [ ]:
df["conc /mM"] = df["Abs_obs"] / epsilon * 1000  # Convert AU to molar and then convert molar to millimolar.

### Inspect the Data

Always make a quick inspection of you data beforte spending any time on analysis. Does it appear to be quality data? Does it appear to match your hypothesis? 

In [ ]:
# Plot the raw data

# get the x and y data
x = df["Time_min"]
y = df["conc /mM"]

# Plot
plt.scatter(                                  # the scatter function will make a classic scatter plot
            x, y,                             # a set of x,y data to plot
            color="navy",                     # many options are available. here I set the color
            s=50,                             # size of the data points. 
            label="Experimental data",        # name of the data set (will be used in a legend)
            )                                 # there are many more option available

plt.axhline(0)                                # Add a line at y=0 (to make a point about data analysis later)

# Styling the Plot
plt.xlabel("Time $/$min")                      # label the axes
plt.ylabel("Conc $/$mM")

plt.title("Simulated First-Order Kinetics")   # add a title to the plot
plt.tight_layout()                            # ensures that the x and y labels are not cut off by the edges

# Output the Plot
plt.savefig("plots/curved1.pdf")                         # write the plot as an image
plt.show()                                    # show the plot

## Using *LMFit*

We are now ready to curve fit the data. We can just use the integrated rate law as-is. I will use the general equation that includes $A_{\infty}$. The beauty of *LMFit* is that we can set $A_\infty$ to zero when we declare the parameters. We coulkd also allow it to be a variable if we wished. I don't need two separate equations so I can include $A_\infty$ or not. I can just set it to a constant when I wish.

In [ ]:
# define the function for the model
def first_order(t, A_0, A_inf, k_obs):
    A_0 = (A_0 - A_inf) * np.exp(-k_obs * t) + A_inf
    return A_0

# Create a model by loading the function via the lmfit.Model tool
model = lmfit.Model(first_order)  

# Set parameters - I can make a variable into a constant by setting vary=False.
params = model.make_params(
    A_0 = dict(
        value = 2.667,  # here I am setting an initial guess. This is not necessary but can help in complex cases.
        vary = flag,    # We should know A_0. We set up the experiment. So fixing the value is appropriate. If flag = True, the value is fixed
        ), 
    A_inf = dict(       # here I am setting A_inf to zero as a constant
        value = 0,      # set A_inf to zero
        vary = flag,    # A_inf should be zero if we are able to subtract the value at the end from the data. Sometimes we cannot wait that long and then we will need to set flag = False and let it vary.
        ), 
    k_obs =  dict(
        value = 0.1, 
        ),
   )

# get the x and y data
x = df["Time_min"]
y = df["conc /mM"]

# use the .fit method on the model object to perform the curve fit
result = model.fit(y, params, t=x) 

print(result.fit_report())
print()
#print(result.ci_report())


### A Quick Plot

The entire curve fit was in the code block above and the visual output of the curve fit is in the code block below. You can see that there is not much typing involved.

In [ ]:
result.plot()
plt.show()

### Calculate Line Fit and Intervals

The code below will make a set of x-data with many points so that we can calculate a smooth curve fit. We will calculate y-values for eact x value using the parameters from the curve fit. *LMFit* can do this from the `result` object using the built-in `result.eval()` function.

*LMFit* can also calculate the standard error at every x-value. Below I use the `result.eval_uncertainty()` function to calculate the $2\sigma$ strandard error (approx. 95% confidence). Adding and subtracting this value from the line fit will give us a range where we would expect the curve fit to be 95% of the time over millions of similar experiments.

The prediction interval is the range in which we would expect experimental data points to appear over many, many repeated experiments. It is obtained by estimating the standard error of the population of the data (a score for how scattered the data points are) and combining that with the errors determined for the fit parameters. It is sometimes useful to present this range on the residual plot to help identify possible outliers.

In [ ]:

# line fit
x_fit = np.linspace(0, np.max(x)+2, 50)     # make an array of 50 points that span the data range
y_fit = result.eval(t=x_fit)                      # the line of the line fit

# confidence interval
y_err = result.eval_uncertainty(t=x_fit, sigma=2) # the 95% confidence interval
ci_upper = y_fit + y_err
ci_lower = y_fit - y_err

# Prediction interval

se = result.eval_uncertainty(t=x_fit, sigma=1)  # standard error of the fit
mse = result.redchi                             # reduced chi-squared value from the fit (mean squared error)
se_pred = np.sqrt(se**2 + mse)

dof = result.ndata - result.nvarys # degrees of freedom = number of data points - number of fitted parameters
tval = t.ppf(0.975,dof)            # two-tailed t-value for 95% confidence interval
pi = tval * se_pred

pi_lower = y_fit - pi
pi_upper = y_fit + pi


## Visualizing the Line Fit

In the code blocks below I will present code to plot the line fit, with its 95% confidence interval; the residual plot with the confidence interval and the prediction interval shown; and a combined plot that includes both the line fit and the residual plot in a single figure. Each plot is output as a pdf file for documents and a png file for web pages.

### Line Fit

The code below will output a plot of the data points, the best-fit line and the $2\sigma$ (~95%) confidence interval for the line fit. 

In [ ]:
# Fancy Plotting
# Note: We have x, y, x_fit, y_fit, ci_upper and ci_lower from previous calculations in the code blocks above.

from matplotlib.ticker import FormatStrFormatter

plt.rcdefaults()                 # reset plotting style
plt.style.use("tufte.mplstyle")  # apply the style sheet

fig, ax = plt.subplots(figsize = (4, 3.5))

ax.scatter(
    x, y, 
    marker = "o", s=32, 
    c="white", edgecolor = "black",
    linewidth = 0.7, 
    label=r"$\log{k_{obs}}$", 
    zorder=3
    )

ax.plot(
    x_fit, y_fit,
    color="black", 
    linewidth=0.5, zorder=2,
    )

ax.fill_between(
    x_fit, ci_lower, ci_upper,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.20,
    label="95% Confidence Band",
    zorder = 2,
    )

ax.spines["left"].set_position(("outward", 8))
ax.spines["bottom"].set_position(("outward", 8))
ax.set(
    ylabel=r"$\text{conc}\;/mM$", 
    xlabel=r"$\text{time }/{min}$",
    xlim=[0,43], 
    xticks = [0,10,20,30,40],                 
    ylim=[-.2,3],     
    yticks = [0,1,2,3],                 
      )
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f')) # 1 decimal place

fig.tight_layout()
fig.savefig("plots/curved2.pdf")

plt.show()



### Residual Plot

The code below will output a plot of the residuals, the $2\sigma$ (~95%) confidence interval for the line fit and the 95% prediction interval for the data set. It is sized to fit the style of my dosuments. You can change the size however you like.

In [ ]:

# Residual Plot

x = result.userkws["t"]      # original x data: we sent in a set labeles as "x"
y = result.data              # original y data
residuals = result.residual  # residuals 

################################################################################

fig, ax = plt.subplots(figsize = (3, 2.5))

ax.scatter(
    x, residuals, 
    marker = "o", s=32, 
    c="white", edgecolor = "black",
    linewidth = 0.7, 
    label=r"$\log{k_{obs}}$", 
    zorder=4
    )

ax.hlines(
    0, np.min(x_fit), np.max(x_fit),
    color="black", 
    linewidth=0.5, zorder=3,
    )

ax.fill_between(
    x_fit, -y_err, y_err,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.20,
    label="95% Confidence Band",
    zorder = 2,
    )

ax.fill_between(
    x_fit, -pi, pi,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.1,
    label="95% prediction Band",
    zorder = 1,
    )

ax.spines["left"].set_position(("outward", 8))
ax.spines["bottom"].set_position(("outward", 8))

res_span = np.max(np.abs(residuals)) * 2
ax.set(ylabel=r"$\Delta\,\text{conc} /mM$", 
       xlabel=r"$\text{time }/{min}^{-1}$",
       xlim=[0,43], 
       xticks = [0,10,20,30,40],                 
#       ylim=[-0.4, 0.4],     
       ylim=[-res_span, res_span],     
#       yticks = [-0.3, 0, 0.3],                 
      )

fig.tight_layout()
fig.savefig("plots/curved3.pdf")

plt.show()



### Combined Plot

The code below will output a plot of the resisuals, the $2\sigma$ (~95%) confidence interval for the line fit and the 95% prediction interval for the data set. It is sized to fit the style of my dosuments. You can change the size however you like.

In [ ]:
from matplotlib.ticker import FormatStrFormatter

plt.rcdefaults()
plt.style.use("tufte.mplstyle")

fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(4,5), height_ratios=[1, 4])  

# plot data points
ax[1].scatter(
    x, y, 
    marker = "o", s=32, 
    c="white", edgecolor = "black",
    linewidth = 0.7, 
    label=r"$\ln{(\text{conc }/M)}$", 
    zorder=3
    )

# plot smooth line fit
ax[1].plot(
    x_fit, y_fit,
    color="black", 
    linewidth=0.5, zorder=2,
    )

# plot confidence interval for line fit
ax[1].fill_between(
    x_fit, ci_lower, ci_upper,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.20,
    label="95% Confidence Band",
    zorder = 2,
    )

# Settings for main plot
ax[1].set(
#   title = "Title",   
    ylabel=r"$\ln{(\text{conc}/mM)}$", 
    xlabel=r"$\text{time }/min^{-1}$",
 #   xlim=[43], 
    xticks = [0,10,20,30,40],                 
 #   ylim=[-2,3],     
 #   yticks = [-8.0,-7.0,-6.0],                 
       )

# add line at x=0 to compare how the value for A_inf sets the endpoint
if False:              # flag to turn this on or off
    ax[1].hlines(
        0, np.min(x_fit), np.max(x_fit),
        color="red", 
        linewidth=0.3, zorder=0,
        )

ax[1].yaxis.set_major_formatter(FormatStrFormatter('%.1f')) # 1 decimal place in y axis

ax[1].spines["left"].set_position(("outward", 8))
ax[1].spines["bottom"].set_position(("outward", 8))

######################
### Plot the residuals
######################

# plot residuas
ax[0].scatter(
    x, residuals, 
    marker = "o", s=32, 
    c="white", edgecolor = "black",
    linewidth = 0.7, 
    label=r"residuals", 
    zorder=4
    )

# make line along zero for residual
ax[0].hlines(
    0, np.min(x_fit), np.max(x_fit),
    color="black", 
    linewidth=0.5, zorder=3,
    )

# add residual confidence interval
ax[0].fill_between(
    x_fit, -y_err, y_err,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.20,
    label="95% Confidence Band",
    zorder = 2,
    )

# add residual prediction interval
ax[0].fill_between(
    x_fit, -pi, pi,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.1,
    label="95% prediction Band",
    zorder = 1,
    )

# settings for residual plot
res_span = np.max(np.abs(residuals)) * 2
ax[0].set(
    ylabel=r"$\text{residuals}$", 
    xlabel=r"",
    xlim=[0,43], 
 #   xticks = [0,5,10,15],                 
    ylim=[-res_span, res_span],
 #   ylim=[-0.4, 0.4],
    yticks = [-0.3, 0, 0.3],                 
    )

ax[0].spines["left"].set_position(("outward", 8))
#ax[0].spines["bottom"].set_position(("outward", 8))
ax[0].set_xticks([])


################################################
### Output Plot
################################################

# Plot as .pdf
fig.savefig(f"plots/curved4.pdf")

### Set face of plot to transparent
ax[0].patch.set_facecolor([0, 0, 0, 0])  
ax[1].patch.set_facecolor([0, 0, 0, 0])  

# Plot as .png with transparent background
fig.savefig(f"plots/curved4.png", dpi=600, 
            facecolor = [0, 0, 0, 0],
        )
# display plot in notebook
plt.show()

print(result.fit_report())
print()
#print(result.ci_report())
